In [5]:
!pip -q install chronos-forecasting==2.3.2 holidays openpyxl pandas numpy matplotlib scikit-learn

In [6]:
from google.colab import files
uploaded = files.upload()

import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import holidays
import torch

warnings.filterwarnings("ignore")

def read_csv_safely(path):
    for encoding in ["utf-8", "utf-8-sig", "latin1"]:
        try:
            return pd.read_csv(path, encoding=encoding)
        except Exception:
            continue
    raise ValueError(f"Could not read {path}")

csv_files = glob.glob("/content/*.csv")
xlsx_files = glob.glob("/content/*.xlsx")

dataset_path = None
hybrid_path = None
excel_path = None

for path in csv_files:
    temp = read_csv_safely(path)
    cols = set(temp.columns)

    if {
        "date", "AP_retail_rs_kg",
        "AP_wholesale_rs_kg", "AP_mandi_modal_rs_kg",
        "AP_arrival_t_day"
    }.issubset(cols):
        dataset_path = path

    elif {
        "timestamp", "predicted_price", "actual_price"
    }.issubset(cols):
        hybrid_path = path

if xlsx_files:
    excel_path = xlsx_files[0]

assert dataset_path, "Historical onion dataset was not identified."
assert hybrid_path, "Hybrid forecast CSV was not identified."
assert excel_path, "Other model Excel file was not identified."

print("Historical dataset:", dataset_path)
print("Hybrid forecast:", hybrid_path)
print("Other model:", excel_path)

raw = read_csv_safely(dataset_path)
hybrid_raw = read_csv_safely(hybrid_path)
excel_raw = pd.read_excel(
    excel_path,
    sheet_name="AP_Daily_Records_Rupees"
)

print("\nFile loading completed.")
print("Historical dataset:", raw.shape)
print("Hybrid forecast:", hybrid_raw.shape)
print("Other model:", excel_raw.shape)

Saving Andhra_Pradesh_Daily_Records_metrics_only (1).xlsx to Andhra_Pradesh_Daily_Records_metrics_only (1) (1).xlsx
Saving final_output.csv to final_output (1).csv
Saving Onion_Daily_Price_Forecasting_Dataset_2022-01-01_to_2026-10-25.csv to Onion_Daily_Price_Forecasting_Dataset_2022-01-01_to_2026-10-25 (1).csv
Historical dataset: /content/Onion_Daily_Price_Forecasting_Dataset_2022-01-01_to_2026-10-25 (1).csv
Hybrid forecast: /content/final_output (1).csv
Other model: /content/Andhra_Pradesh_Daily_Records_metrics_only (1).xlsx

File loading completed.
Historical dataset: (1759, 17)
Hybrid forecast: (365, 81)
Other model: (730, 8)


In [ ]:
# ============================================================
# CELL 1A — NORMALIZE THE THREE INPUT DATAFRAMES
# Run immediately after the file-loading cell.
# ============================================================

import pandas as pd
import numpy as np

# Helper: normalize date/timestamp columns to timezone-naive daily dates.
def normalize_daily_dates(values):
    return (
        pd.to_datetime(values, errors="coerce", utc=True)
        .dt.tz_localize(None)
        .dt.normalize()
    )

# ------------------------------------------------------------
# 1. HYBRID MODEL CSV
# Raw variables created in the file-loading cell: hybrid_raw
# Required source fields: timestamp, predicted_price
# ------------------------------------------------------------
required_hybrid_columns = ["timestamp", "predicted_price"]
missing = [c for c in required_hybrid_columns if c not in hybrid_raw.columns]
if missing:
    raise ValueError(f"Hybrid CSV is missing required columns: {missing}")

hybrid_source = hybrid_raw.copy()

# Keep only the AP retail-price forecast/model where metadata exists.
if "target_field" in hybrid_source.columns:
    hybrid_source = hybrid_source[
        hybrid_source["target_field"].astype(str).str.strip().eq("AP_retail_rs_kg")
    ].copy()

if "model" in hybrid_source.columns:
    hybrid_source = hybrid_source[
        hybrid_source["model"].astype(str).str.strip().eq("prophet_lightgbm_hybrid")
    ].copy()

hybrid = hybrid_source[["timestamp", "predicted_price"]].copy()
hybrid["date"] = normalize_daily_dates(hybrid["timestamp"])
hybrid["Hybrid_Prediction"] = pd.to_numeric(
    hybrid["predicted_price"], errors="coerce"
)
hybrid = hybrid[["date", "Hybrid_Prediction"]]
hybrid = hybrid.dropna(subset=["date", "Hybrid_Prediction"])
hybrid = (
    hybrid.sort_values("date")
    .drop_duplicates(subset=["date"], keep="last")
    .reset_index(drop=True)
)

if hybrid.empty:
    raise ValueError("No valid AP retail predictions were found in the Hybrid CSV.")

# ------------------------------------------------------------
# 2. OTHER MODEL EXCEL
# Raw variable created in the file-loading cell: excel_raw
# Required source fields: Date, Retail Price (₹/kg)
# ------------------------------------------------------------
required_excel_columns = ["Date", "Retail Price (₹/kg)"]
missing = [c for c in required_excel_columns if c not in excel_raw.columns]
if missing:
    raise ValueError(f"Other-model Excel sheet is missing required columns: {missing}")

excel = excel_raw[["Date", "Retail Price (₹/kg)"]].copy()
excel["date"] = normalize_daily_dates(excel["Date"])
excel["Other_Model_Prediction"] = pd.to_numeric(
    excel["Retail Price (₹/kg)"], errors="coerce"
)
excel = excel[["date", "Other_Model_Prediction"]]
excel = excel.dropna(subset=["date", "Other_Model_Prediction"])
excel = (
    excel.sort_values("date")
    .drop_duplicates(subset=["date"], keep="last")
    .reset_index(drop=True)
)

if excel.empty:
    raise ValueError("No valid retail-price predictions were found in the Excel sheet.")

# ------------------------------------------------------------
# 3. ACTUAL RETAIL PRICE SERIES
# Used by the forecast-overlap audit cell.
# ------------------------------------------------------------
if not {"date", "AP_retail_rs_kg"}.issubset(raw.columns):
    raise ValueError("Historical dataset needs date and AP_retail_rs_kg columns.")

actual_series = raw[["date", "AP_retail_rs_kg"]].copy()
actual_series["date"] = normalize_daily_dates(actual_series["date"])
actual_series["AP_retail_rs_kg"] = pd.to_numeric(
    actual_series["AP_retail_rs_kg"], errors="coerce"
)
actual_series = actual_series.dropna(subset=["date", "AP_retail_rs_kg"])
actual_series = (
    actual_series.sort_values("date")
    .drop_duplicates(subset=["date"], keep="last")
    .reset_index(drop=True)
)

print("INPUT DATAFRAMES PREPARED")
print("-------------------------")
print("Historical actuals:", actual_series.shape)
print("Hybrid forecasts:", hybrid.shape)
print("Other-model forecasts:", excel.shape)
print("Hybrid date range:", hybrid["date"].min(), "to", hybrid["date"].max())
print("Other-model date range:", excel["date"].min(), "to", excel["date"].max())



In [7]:
DATE_COL = "date"
TARGET = "AP_retail_rs_kg"

PRICE_COLS = [
    "AP_retail_rs_kg",
    "AP_wholesale_rs_kg",
    "AP_mandi_modal_rs_kg"
]

PAST_COVARIATES = [
    "AP_wholesale_rs_kg",
    "AP_mandi_modal_rs_kg",
    "AP_arrival_t_day",
    "AP_arrival_missing"
]

required_columns = [
    DATE_COL,
    TARGET,
    "AP_wholesale_rs_kg",
    "AP_mandi_modal_rs_kg",
    "AP_arrival_t_day"
]

missing_columns = [
    c for c in required_columns if c not in raw.columns
]

if missing_columns:
    raise ValueError(f"Missing dataset columns: {missing_columns}")

data = raw[required_columns].copy()

data[DATE_COL] = pd.to_datetime(
    data[DATE_COL],
    errors="coerce"
)

data = data.dropna(subset=[DATE_COL])
data = data.sort_values(DATE_COL)
data = data.drop_duplicates(
    subset=[DATE_COL],
    keep="last"
)

data = data.set_index(DATE_COL)

for col in required_columns[1:]:
    data[col] = pd.to_numeric(
        data[col],
        errors="coerce"
    )

# Zero arrival quantity is treated as missing.
data["AP_arrival_t_day"] = (
    data["AP_arrival_t_day"].replace(0, np.nan)
)

# Record which arrival observations were originally missing.
data["AP_arrival_missing"] = (
    data["AP_arrival_t_day"].isna().astype(float)
)

# Create a complete daily date index.
daily_index = pd.date_range(
    start=data.index.min(),
    end=data.index.max(),
    freq="D"
)

data = data.reindex(daily_index)
data.index.name = DATE_COL

# Interpolate only short internal gaps in price columns.
for col in PRICE_COLS:
    data[col] = data[col].interpolate(
        method="time",
        limit=7,
        limit_area="inside"
    )

# Interpolate only short internal gaps in arrival quantity.
data["AP_arrival_t_day"] = data["AP_arrival_t_day"].interpolate(
    method="time",
    limit=7,
    limit_area="inside"
)

# Preserve missingness information.
# Dates originally missing or zero remain marked as missing.
data["AP_arrival_missing"] = (
    data["AP_arrival_missing"].fillna(1.0)
)

print("Date range:", data.index.min().date(), "to", data.index.max().date())
print("Total daily rows:", len(data))

print("\nMissing values after short-gap interpolation:")
print(data[PRICE_COLS + ["AP_arrival_t_day"]].isna().sum())

print("\nArrival missingness indicator:")
print(data["AP_arrival_missing"].value_counts())

# Price data must be complete.
if data[PRICE_COLS].isna().any().any():
    raise ValueError(
        "Price columns still contain missing values. "
        "Inspect the gaps; they have not been silently filled."
    )

if (data[TARGET] <= 0).any():
    raise ValueError("Retail price contains zero or negative values.")

if data["AP_arrival_t_day"].notna().sum() == 0:
    raise ValueError("No valid arrival quantity observations exist.")

print("\nCell 3 completed successfully.")

Date range: 2022-01-01 to 2026-10-25
Total daily rows: 1759

Missing values after short-gap interpolation:
AP_retail_rs_kg           0
AP_wholesale_rs_kg        0
AP_mandi_modal_rs_kg      0
AP_arrival_t_day        103
dtype: int64

Arrival missingness indicator:
AP_arrival_missing
0.0    1268
1.0     491
Name: count, dtype: int64

Cell 3 completed successfully.


In [8]:
def add_calendar_features(index):

    idx = pd.DatetimeIndex(index)

    years = range(
        idx.min().year,
        idx.max().year + 2
    )

    india_holidays = holidays.country_holidays(
        "IN",
        years=years
    )

    result = pd.DataFrame(index=idx)

    result["dow"] = idx.dayofweek.astype(float)
    result["month"] = idx.month.astype(float)
    result["day"] = idx.day.astype(float)
    result["quarter"] = idx.quarter.astype(float)

    result["is_weekend"] = (
        idx.dayofweek >= 5
    ).astype(float)

    result["is_holiday"] = [
        float(d.date() in india_holidays)
        for d in idx
    ]

    day_of_year = idx.dayofyear.to_numpy()

    result["year_sin"] = np.sin(
        2 * np.pi * day_of_year / 365.25
    )

    result["year_cos"] = np.cos(
        2 * np.pi * day_of_year / 365.25
    )

    result.index.name = "date"

    return result


CALENDAR_COLS = [
    "dow",
    "month",
    "day",
    "quarter",
    "is_weekend",
    "is_holiday",
    "year_sin",
    "year_cos"
]

print("Calendar features created:", CALENDAR_COLS)

Calendar features created: ['dow', 'month', 'day', 'quarter', 'is_weekend', 'is_holiday', 'year_sin', 'year_cos']


In [9]:
from chronos import Chronos2Pipeline

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)

pipeline = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map=DEVICE
)

print("Chronos-2 loaded successfully.")

Device: cuda


config.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  478MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Chronos-2 loaded successfully.


In [10]:
QUANTILES = [0.1, 0.5, 0.9]

def run_chronos_forecast(origin, horizon):

    origin = pd.Timestamp(origin)

    history = data.loc[:origin].copy()

    if len(history) < 365:
        raise ValueError(
            f"Insufficient historical data at {origin.date()}"
        )

    # Fill remaining arrival gaps using only information
    # available up to this forecast origin.
    arrival_median = history["AP_arrival_t_day"].median()

    if pd.isna(arrival_median):
        raise ValueError(
            f"No valid historical arrival data at {origin.date()}"
        )

    history["AP_arrival_t_day"] = (
        history["AP_arrival_t_day"].fillna(arrival_median)
    )

    future_dates = pd.date_range(
        start=origin + pd.Timedelta(days=1),
        periods=horizon,
        freq="D"
    )

    historical_calendar = add_calendar_features(
        history.index
    )

    context = pd.concat(
        [history, historical_calendar],
        axis=1
    ).reset_index()

    context["item_id"] = "AP_ONION"

    future_calendar = add_calendar_features(
        future_dates
    ).reset_index()

    future_calendar["item_id"] = "AP_ONION"

    context = context.rename(
        columns={DATE_COL: "timestamp"}
    )

    future_calendar = future_calendar.rename(
        columns={DATE_COL: "timestamp"}
    )

    context = context[
        [
            "item_id",
            "timestamp",
            TARGET
        ] + PAST_COVARIATES + CALENDAR_COLS
    ]

    future_df = future_calendar[
        [
            "item_id",
            "timestamp"
        ] + CALENDAR_COLS
    ]

    forecast = pipeline.predict_df(
        df=context,
        future_df=future_df,
        id_column="item_id",
        timestamp_column="timestamp",
        target=TARGET,
        prediction_length=horizon,
        quantile_levels=QUANTILES,
        freq="D"
    )

    forecast = forecast.rename(
        columns={
            "timestamp": "date",
            "0.1": "P10",
            "0.5": "P50",
            "0.9": "P90"
        }
    )

    forecast["origin"] = origin
    forecast["horizon_days"] = horizon
    forecast["days_ahead"] = np.arange(1, horizon + 1)

    return forecast[
        [
            "origin",
            "date",
            "horizon_days",
            "days_ahead",
            "P10",
            "P50",
            "P90"
        ]
    ].copy()

In [11]:
HORIZONS = [90, 180, 365]

ORIGIN_STEP_DAYS = 30
MIN_HISTORY_DAYS = 365

dataset_end = data.index.max()

all_forecasts = []

for horizon in HORIZONS:

    first_origin = (
        data.index.min()
        + pd.Timedelta(days=MIN_HISTORY_DAYS)
    )

    last_origin = (
        dataset_end
        - pd.Timedelta(days=horizon)
    )

    origins = pd.date_range(
        start=first_origin,
        end=last_origin,
        freq=f"{ORIGIN_STEP_DAYS}D"
    )

    print(
        f"\nRunning {horizon}-day backtest. "
        f"Origins: {len(origins)}"
    )

    for i, origin in enumerate(origins, start=1):

        print(
            f"{i}/{len(origins)} — {origin.date()}"
        )

        forecast = run_chronos_forecast(
            origin,
            horizon
        )

        actual = (
            data[TARGET]
            .rename("actual_price")
            .reset_index()
        )

        forecast = forecast.merge(
            actual,
            left_on="date",
            right_on=DATE_COL,
            how="left"
        )

        forecast = forecast.drop(
            columns=[DATE_COL]
        )

        all_forecasts.append(forecast)

backtest = pd.concat(
    all_forecasts,
    ignore_index=True
)

backtest = backtest.dropna(
    subset=["actual_price"]
)

print("\nBacktesting completed.")
print("Total forecast observations:", len(backtest))

print(
    backtest.groupby("horizon_days")["origin"]
    .nunique()
)


Running 90-day backtest. Origins: 44
1/44 — 2023-01-01
2/44 — 2023-01-31
3/44 — 2023-03-02
4/44 — 2023-04-01
5/44 — 2023-05-01
6/44 — 2023-05-31
7/44 — 2023-06-30
8/44 — 2023-07-30
9/44 — 2023-08-29
10/44 — 2023-09-28
11/44 — 2023-10-28
12/44 — 2023-11-27
13/44 — 2023-12-27
14/44 — 2024-01-26
15/44 — 2024-02-25
16/44 — 2024-03-26
17/44 — 2024-04-25
18/44 — 2024-05-25
19/44 — 2024-06-24
20/44 — 2024-07-24
21/44 — 2024-08-23
22/44 — 2024-09-22
23/44 — 2024-10-22
24/44 — 2024-11-21
25/44 — 2024-12-21
26/44 — 2025-01-20
27/44 — 2025-02-19
28/44 — 2025-03-21
29/44 — 2025-04-20
30/44 — 2025-05-20
31/44 — 2025-06-19
32/44 — 2025-07-19
33/44 — 2025-08-18
34/44 — 2025-09-17
35/44 — 2025-10-17
36/44 — 2025-11-16
37/44 — 2025-12-16
38/44 — 2026-01-15
39/44 — 2026-02-14
40/44 — 2026-03-16
41/44 — 2026-04-15
42/44 — 2026-05-15
43/44 — 2026-06-14
44/44 — 2026-07-14

Running 180-day backtest. Origins: 41
1/41 — 2023-01-01
2/41 — 2023-01-31
3/41 — 2023-03-02
4/41 — 2023-04-01
5/41 — 2023-05-01
6/41 —

In [12]:
unique_origins = sorted(
    backtest["origin"].drop_duplicates()
)

if len(unique_origins) < 6:
    raise ValueError(
        "Not enough forecast origins for a reliable split."
    )

split_position = int(
    len(unique_origins) * 0.60
)

calibration_origins = set(
    unique_origins[:split_position]
)

validation_origins = set(
    unique_origins[split_position:]
)

cal = backtest[
    backtest["origin"].isin(calibration_origins)
].copy()

val = backtest[
    backtest["origin"].isin(validation_origins)
].copy()


def nonconformity_score(df):

    left_width = np.maximum(
        df["P50"] - df["P10"],
        1e-8
    )

    right_width = np.maximum(
        df["P90"] - df["P50"],
        1e-8
    )

    left_score = (
        df["P50"] - df["actual_price"]
    ) / left_width

    right_score = (
        df["actual_price"] - df["P50"]
    ) / right_width

    return np.maximum.reduce([
        np.zeros(len(df)),
        left_score.to_numpy(),
        right_score.to_numpy()
    ])


cal["score"] = nonconformity_score(cal)

TARGET_COVERAGE = 0.80

n = len(cal)

rank = int(
    np.ceil((n + 1) * TARGET_COVERAGE)
)

rank = min(max(rank, 1), n)

GLOBAL_K = float(
    np.sort(cal["score"].to_numpy())[rank - 1]
)

print("Calibration origins:", len(calibration_origins))
print("Validation origins:", len(validation_origins))
print("Calibration observations:", n)
print("Order statistic rank:", rank)
print("Target coverage:", TARGET_COVERAGE)
print("\nGLOBAL K:", GLOBAL_K)

Calibration origins: 26
Validation origins: 18
Calibration observations: 16510
Order statistic rank: 13209
Target coverage: 0.8

GLOBAL K: 1.266404067979788


In [13]:
val["P10_cal"] = (
    val["P50"]
    - GLOBAL_K * (val["P50"] - val["P10"])
)

val["P90_cal"] = (
    val["P50"]
    + GLOBAL_K * (val["P90"] - val["P50"])
)

val["raw_hit"] = (
    (val["actual_price"] >= val["P10"]) &
    (val["actual_price"] <= val["P90"])
)

val["calibrated_hit"] = (
    (val["actual_price"] >= val["P10_cal"]) &
    (val["actual_price"] <= val["P90_cal"])
)


def metric_row(df, label):

    actual = df["actual_price"].to_numpy()
    predicted = df["P50"].to_numpy()

    mae = np.mean(
        np.abs(actual - predicted)
    )

    rmse = np.sqrt(
        np.mean((actual - predicted) ** 2)
    )

    mape = np.mean(
        np.abs(
            (actual - predicted)
            / np.maximum(np.abs(actual), 1e-8)
        )
    ) * 100

    return {
        "Horizon": label,
        "Forecast rows": len(df),
        "Origins": df["origin"].nunique(),
        "MAE": mae,
        "RMSE": rmse,
        "MAPE (%)": mape,
        "Original coverage (%)": df["raw_hit"].mean() * 100,
        "Calibrated coverage (%)": df["calibrated_hit"].mean() * 100
    }


metrics = pd.DataFrame([
    metric_row(val, "Overall")
])

for horizon, group in val.groupby("horizon_days"):

    metrics.loc[len(metrics)] = metric_row(
        group,
        f"{horizon} days"
    )

display(metrics.round(3))

,Horizon,Forecast rows,Origins,MAE,RMSE,MAPE (%),Original coverage (%),Calibrated coverage (%)
0,Overall,7605,18,3.930,6.100,14.976,79.421,90.204
1,90 days,1620,18,2.801,4.475,9.888,80.864,90.864
2,180 days,2700,15,3.524,5.551,13.250,80.889,90.815
3,365 days,3285,9,4.822,7.139,18.903,77.504,89.376


In [14]:
k_evidence = pd.DataFrame([{
    "Global K": GLOBAL_K,
    "Target coverage": TARGET_COVERAGE,
    "Calibration observations": n,
    "Order statistic rank": rank,
    "Calibration origins": len(calibration_origins),
    "Validation origins": len(validation_origins),
    "Overall validation coverage (%)": (
        val["calibrated_hit"].mean() * 100
    ),
    "Validation status": (
        "PASSED"
        if val["calibrated_hit"].mean() >= TARGET_COVERAGE
        else "NOT PASSED"
    )
}])

display(k_evidence)

val.to_csv(
    "/content/Chronos2_Validation_Forecasts.csv",
    index=False
)

metrics.to_csv(
    "/content/Chronos2_Validation_Metrics.csv",
    index=False
)

k_evidence.to_csv(
    "/content/Chronos2_Global_K_Evidence.csv",
    index=False
)

,Global K,Target coverage,Calibration observations,Order statistic rank,Calibration origins,Validation origins,Overall validation coverage (%),Validation status
0,1.266404,0.8,16510,13209,26,18,90.203813,PASSED


In [16]:
print("Columns in val:")
print(val.columns.tolist())

print("\nFirst 5 rows:")
display(val.head())

Columns in val:
['origin', 'horizon_days', 'days_ahead', 'P10', 'P50', 'P90', 'actual_price', 'P10_cal', 'P90_cal', 'raw_hit', 'calibrated_hit']

First 5 rows:


,origin,horizon_days,days_ahead,P10,P50,P90,actual_price,P10_cal,P90_cal,raw_hit,calibrated_hit
2340,2025-02-19,90,1,35.626148,37.288364,38.837353,39.0000,35.183327,39.250008,False,True
2341,2025-02-19,90,2,35.173813,37.063213,38.783123,39.0000,34.670467,39.241314,False,True
2342,2025-02-19,90,3,34.820667,36.955780,38.873333,39.0000,34.251865,39.384178,False,True
2343,2025-02-19,90,4,34.436501,36.850124,38.959618,34.6667,33.793503,39.521595,True,True
2344,2025-02-19,90,5,34.222374,36.769032,39.123989,38.5000,33.543934,39.751358,True,True


In [18]:
# ============================================================
# CELL 11 — PREPARE VALIDATION RESULTS FOR MODEL COMPARISON
# ============================================================

# Reconstruct forecast date from origin + days_ahead.
# The forecast date is the origin date plus the forecast lead.

val = val.copy()

val["origin"] = pd.to_datetime(val["origin"], errors="coerce")

val["days_ahead"] = pd.to_numeric(
    val["days_ahead"], errors="coerce"
)

if val["origin"].isna().any():
    raise ValueError("Some validation origin dates could not be parsed.")

if val["days_ahead"].isna().any():
    raise ValueError("Some days_ahead values could not be parsed.")

val["date"] = (
    val["origin"] +
    pd.to_timedelta(val["days_ahead"], unit="D")
)

val["date"] = pd.to_datetime(val["date"]).dt.normalize()

# Sort validation forecasts
val = val.sort_values(
    ["date", "days_ahead"]
).reset_index(drop=True)

print("Chronos validation dates reconstructed successfully.")
print("Validation rows:", len(val))
print("Date range:", val["date"].min(), "to", val["date"].max())

display(
    val[
        ["origin", "days_ahead", "date",
         "actual_price", "P50", "P10_cal", "P90_cal"]
    ].head()
)

Chronos validation dates reconstructed successfully.
Validation rows: 7605
Date range: 2025-02-20 00:00:00 to 2026-10-17 00:00:00


,origin,days_ahead,date,actual_price,P50,P10_cal,P90_cal
0,2025-02-19,1,2025-02-20,39.0,37.288364,35.183327,39.250008
1,2025-02-19,1,2025-02-20,39.0,37.270100,35.243435,39.128357
2,2025-02-19,1,2025-02-20,39.0,37.247005,35.262333,39.102528
3,2025-02-19,2,2025-02-21,39.0,37.063213,34.670467,39.241314
4,2025-02-19,2,2025-02-21,39.0,37.031235,34.785770,39.105030


In [ ]:
# Find loaded DataFrames in the current notebook session

import pandas as pd

dataframes = {
    name: obj
    for name, obj in globals().items()
    if isinstance(obj, pd.DataFrame)
}

print("DataFrames currently available:\n")

for name, df_obj in dataframes.items():
    print(f"\nVariable: {name}")
    print("Shape:", df_obj.shape)
    print("Columns:", df_obj.columns.tolist()[:15])
    print(df_obj.head(2))

In [23]:
# ============================================================
# CELL 11B — EXPLORATORY ROLLING-ORIGIN COMPARISON DATA
# NOTE: This section is diagnostic only. Use the controlled common-origin
# comparison at the end for the final three-model performance ranking.
# ============================================================

# 1. Prepare Chronos validation predictions
chronos_compare = val[
    ["date", "origin", "horizon_days", "days_ahead",
     "actual_price", "P50", "P10_cal", "P90_cal"]
].copy()

chronos_compare["date"] = pd.to_datetime(
    chronos_compare["date"]
).dt.normalize()

chronos_compare = chronos_compare.rename(columns={
    "actual_price": "Chronos_Backtest_Actual",
    "P50": "Chronos",
    "P10_cal": "Chronos_P10",
    "P90_cal": "Chronos_P90"
})

# 2. Prepare historical actual prices
actual_compare = raw[
    ["date", "AP_retail_rs_kg"]
].copy()

actual_compare["date"] = pd.to_datetime(
    actual_compare["date"],
    errors="coerce"
).dt.normalize()

actual_compare["AP_retail_rs_kg"] = pd.to_numeric(
    actual_compare["AP_retail_rs_kg"],
    errors="coerce"
)

actual_compare = actual_compare.rename(columns={
    "AP_retail_rs_kg": "Actual"
})

# 3. Prepare Hybrid predictions
hybrid_compare = hybrid[
    ["date", "Hybrid_Prediction"]
].copy()

hybrid_compare["date"] = pd.to_datetime(
    hybrid_compare["date"],
    errors="coerce",
    utc=True
).dt.tz_localize(None).dt.normalize()

# 4. Prepare Excel model predictions
excel_compare = excel[
    ["date", "Other_Model_Prediction"]
].copy()

excel_compare["date"] = pd.to_datetime(
    excel_compare["date"],
    errors="coerce"
).dt.normalize()

# 5. Merge all datasets
comparison = (
    chronos_compare
    .merge(actual_compare, on="date", how="left")
    .merge(hybrid_compare, on="date", how="left")
    .merge(excel_compare, on="date", how="left")
)

# 6. Keep rows with an actual historical price
comparison = comparison.dropna(subset=["Actual"])

# 7. Calculate absolute errors
comparison["Chronos_Abs_Error"] = (
    comparison["Chronos"] - comparison["Actual"]
).abs()

comparison["Hybrid_Abs_Error"] = (
    comparison["Hybrid_Prediction"] - comparison["Actual"]
).abs()

comparison["Other_Model_Abs_Error"] = (
    comparison["Other_Model_Prediction"] - comparison["Actual"]
).abs()

# 8. Identify closest prediction
error_columns = {
    "Chronos": "Chronos_Abs_Error",
    "Hybrid": "Hybrid_Abs_Error",
    "Other Model": "Other_Model_Abs_Error"
}

def find_closest_model(row):
    available = {
        model: row[column]
        for model, column in error_columns.items()
        if pd.notna(row[column])
    }

    if not available:
        return "No prediction"

    minimum = min(available.values())

    winners = [
        model for model, error in available.items()
        if np.isclose(error, minimum)
    ]

    return " / ".join(winners)

comparison["Closest_Model"] = comparison.apply(
    find_closest_model,
    axis=1
)

# 9. Display results
comparison = comparison.sort_values(
    ["date", "days_ahead"]
).reset_index(drop=True)

print("MODEL COMPARISON DATASET CREATED")
print("--------------------------------")
print("Total comparison rows:", len(comparison))
print("Date range:", comparison["date"].min(),
      "to", comparison["date"].max())

print("\nAvailable predictions:")
print("Chronos:", comparison["Chronos"].notna().sum())
print("Hybrid:", comparison["Hybrid_Prediction"].notna().sum())
print("Other Model:", comparison["Other_Model_Prediction"].notna().sum())

display(comparison.head(15))

MODEL COMPARISON DATASET CREATED
--------------------------------
Total comparison rows: 7605
Date range: 2025-02-20 00:00:00 to 2026-10-17 00:00:00

Available predictions:
Chronos: 7605
Hybrid: 3555
Other Model: 3573


,date,origin,horizon_days,days_ahead,Chronos_Backtest_Actual,Chronos,Chronos_P10,Chronos_P90,Actual,Hybrid_Prediction,Other_Model_Prediction,Chronos_Abs_Error,Hybrid_Abs_Error,Other_Model_Abs_Error,Closest_Model
0,2025-02-20,2025-02-19,90,1,39.0000,37.288364,35.183327,39.250008,39.0000,NaN,NaN,1.711636,NaN,NaN,Chronos
1,2025-02-20,2025-02-19,180,1,39.0000,37.270100,35.243435,39.128357,39.0000,NaN,NaN,1.729900,NaN,NaN,Chronos
2,2025-02-20,2025-02-19,365,1,39.0000,37.247005,35.262333,39.102528,39.0000,NaN,NaN,1.752995,NaN,NaN,Chronos
3,2025-02-21,2025-02-19,90,2,39.0000,37.063213,34.670467,39.241314,39.0000,NaN,NaN,1.936787,NaN,NaN,Chronos
4,2025-02-21,2025-02-19,180,2,39.0000,37.031235,34.785770,39.105030,39.0000,NaN,NaN,1.968765,NaN,NaN,Chronos
5,2025-02-21,2025-02-19,365,2,39.0000,37.005905,34.800495,39.093819,39.0000,NaN,NaN,1.994095,NaN,NaN,Chronos
6,2025-02-22,2025-02-19,90,3,39.0000,36.955780,34.251865,39.384178,39.0000,NaN,NaN,2.044220,NaN,NaN,Chronos
7,2025-02-22,2025-02-19,180,3,39.0000,36.898609,34.293854,39.245087,39.0000,NaN,NaN,2.101391,NaN,NaN,Chronos
8,2025-02-22,2025-02-19,365,3,39.0000,36.904037,34.342300,39.242714,39.0000,NaN,NaN,2.095963,NaN,NaN,Chronos
9,2025-02-23,2025-02-19,90,4,34.6667,36.850124,33.793503,39.521595,34.6667,NaN,NaN,2.183424,NaN,NaN,Chronos


In [24]:
# ============================================================
# CHRONOS-2 ROLLING-ORIGIN BACKTEST METRICS BY HORIZON
# This is a Chronos-only validation diagnostic, NOT the final
# three-model head-to-head comparison.
# ============================================================

import numpy as np
import pandas as pd

scorecard = []

for horizon in [90, 180, 365]:
    subset = val[val["horizon_days"] == horizon].copy()
    evaluation = subset[["actual_price", "P50"]].dropna()

    if evaluation.empty:
        result = {
            "Observations": 0,
            "MAE": np.nan,
            "RMSE": np.nan,
            "MAPE (%)": np.nan
        }
    else:
        errors = evaluation["P50"] - evaluation["actual_price"]
        result = {
            "Observations": len(evaluation),
            "MAE": errors.abs().mean(),
            "RMSE": np.sqrt((errors ** 2).mean()),
            "MAPE (%)": (
                errors.abs()
                / evaluation["actual_price"].abs().replace(0, np.nan)
            ).mean() * 100
        }

    scorecard.append({
        "Model": "Chronos-2",
        "Evaluation": f"Rolling-origin {horizon}-day validation",
        **result
    })

scorecard_df = pd.DataFrame(scorecard)

print("CHRONOS-2 ROLLING-ORIGIN VALIDATION — BY HORIZON")
print("These metrics evaluate Chronos only. Do not rank these values")
print("against Hybrid/Other Model metrics from a different forecast origin.")
display(scorecard_df.round(3))



MODEL PERFORMANCE SCORECARD


,Model,Evaluation,Observations,MAE,RMSE,MAPE (%)
0,Chronos-2,90-day horizon,1620,2.801,4.475,9.888
1,Chronos-2,180-day horizon,2700,3.524,5.551,13.250
2,Chronos-2,365-day horizon,3285,4.822,7.139,18.903
3,Hybrid,Available forecast overlap,290,3.019,3.631,9.496
4,Other Model,Available forecast overlap,291,23.717,25.877,93.679



THREE-MODEL COMPARISON
Common dates: 290


,Model,Closest Predictions
0,Chronos-2,182
1,Hybrid,68
2,Other Model,40


In [25]:
# ============================================================
# CELL 13 — FORECAST ORIGIN AND OVERLAP AUDIT
# ============================================================

print("CHRONOS VALIDATION ORIGINS")
print("--------------------------")

print(
    val.groupby("horizon_days")["origin"]
    .agg(["min", "max", "nunique"])
)

print("\nHYBRID FORECAST DETAILS")
print("----------------------")
print("First date:", hybrid["date"].min())
print("Last date:", hybrid["date"].max())
print("Number of dates:", hybrid["date"].nunique())

print("\nOTHER MODEL FORECAST DETAILS")
print("----------------------------")
print("First date:", excel["date"].min())
print("Last date:", excel["date"].max())
print("Number of dates:", excel["date"].nunique())

print("\nCHRONOS UNIQUE VALIDATION DATES")
print("--------------------------------")
print("First date:", val["date"].min())
print("Last date:", val["date"].max())
print("Unique dates:", val["date"].nunique())

print("\nCOMMON DATE RANGE")
print("-----------------")

common_dates = (
    set(pd.to_datetime(hybrid["date"]).dt.normalize())
    & set(pd.to_datetime(excel["date"]).dt.normalize())
    & set(pd.to_datetime(actual_series["date"]).dt.normalize())
)

print("Dates available in all three datasets:", len(common_dates))

if common_dates:
    print("First common date:", min(common_dates))
    print("Last common date:", max(common_dates))

print("\nCHRONOS ORIGIN COUNTS")
display(
    val.groupby(["horizon_days", "origin"])
    .size()
    .reset_index(name="Forecast_Rows")
    .groupby("horizon_days")
    .agg(
        Origins=("origin", "nunique"),
        Total_Rows=("Forecast_Rows", "sum")
    )
)

CHRONOS VALIDATION ORIGINS
--------------------------
                    min        max  nunique
horizon_days                               
90           2025-02-19 2026-07-14       18
180          2025-02-19 2026-04-15       15
365          2025-02-19 2025-10-17        9

HYBRID FORECAST DETAILS
----------------------
First date: 2026-01-01 00:00:00+00:00
Last date: 2026-12-31 00:00:00+00:00
Number of dates: 365

OTHER MODEL FORECAST DETAILS
----------------------------
First date: 2025-12-31 00:00:00
Last date: 2027-12-30 00:00:00
Number of dates: 730

CHRONOS UNIQUE VALIDATION DATES
--------------------------------
First date: 2025-02-20 00:00:00
Last date: 2026-10-17 00:00:00
Unique dates: 605

COMMON DATE RANGE
-----------------
Dates available in all three datasets: 0

CHRONOS ORIGIN COUNTS


,Origins,Total_Rows
horizon_days,,
90,18,1620
180,15,2700
365,9,3285


In [26]:
# ============================================================
# CELL 14 — CONTROLLED BACKTEST DATA AUDIT
# ============================================================

# 1. Historical actuals
historical = raw[["date", "AP_retail_rs_kg"]].copy()

historical["date"] = pd.to_datetime(
    historical["date"], errors="coerce"
).dt.normalize()

historical["AP_retail_rs_kg"] = pd.to_numeric(
    historical["AP_retail_rs_kg"], errors="coerce"
)

historical = historical.dropna(
    subset=["date", "AP_retail_rs_kg"]
)

origin_date = pd.Timestamp("2025-12-31")
evaluation_start = pd.Timestamp("2026-01-01")
evaluation_end = pd.Timestamp("2026-12-31")

history_at_origin = historical[
    historical["date"] <= origin_date
].copy()

actual_2026 = historical[
    historical["date"].between(
        evaluation_start, evaluation_end
    )
].copy()

# 2. Hybrid forecasts
hybrid_2026 = hybrid.copy()

hybrid_2026["date"] = pd.to_datetime(
    hybrid_2026["date"], utc=True
).dt.tz_localize(None).dt.normalize()

hybrid_2026 = hybrid_2026[
    hybrid_2026["date"].between(
        evaluation_start, evaluation_end
    )
].copy()

# 3. Other model forecasts
excel_2026 = excel.copy()

excel_2026["date"] = pd.to_datetime(
    excel_2026["date"], errors="coerce"
).dt.normalize()

excel_2026 = excel_2026[
    excel_2026["date"].between(
        evaluation_start, evaluation_end
    )
].copy()

# 4. Report
print("CONTROLLED BACKTEST AUDIT")
print("=========================")

print("\nHistorical data available at forecast origin:")
print("Rows:", len(history_at_origin))
print("First date:", history_at_origin["date"].min())
print("Last date:", history_at_origin["date"].max())

print("\nActual 2026 observations:")
print("Rows:", len(actual_2026))
print("First date:", actual_2026["date"].min())
print("Last date:", actual_2026["date"].max())

print("\nHybrid 2026 forecasts:")
print("Rows:", len(hybrid_2026))
print("First date:", hybrid_2026["date"].min())
print("Last date:", hybrid_2026["date"].max())

print("\nOther model 2026 forecasts:")
print("Rows:", len(excel_2026))
print("First date:", excel_2026["date"].min())
print("Last date:", excel_2026["date"].max())

print("\nMissing values:")
print("Historical:", history_at_origin["AP_retail_rs_kg"].isna().sum())
print("Hybrid:", hybrid_2026["Hybrid_Prediction"].isna().sum())
print("Other:", excel_2026["Other_Model_Prediction"].isna().sum())

CONTROLLED BACKTEST AUDIT

Historical data available at forecast origin:
Rows: 1461
First date: 2022-01-01 00:00:00
Last date: 2025-12-31 00:00:00

Actual 2026 observations:
Rows: 298
First date: 2026-01-01 00:00:00
Last date: 2026-10-25 00:00:00

Hybrid 2026 forecasts:
Rows: 365
First date: 2026-01-01 00:00:00
Last date: 2026-12-31 00:00:00

Other model 2026 forecasts:
Rows: 365
First date: 2026-01-01 00:00:00
Last date: 2026-12-31 00:00:00

Missing values:
Historical: 0
Hybrid: 0
Other: 0


In [27]:
# ============================================================
# CELL 15 — CHRONOS-2 CONTROLLED 365-DAY FORECAST
# Forecast origin: 31 December 2025
# ============================================================

import pandas as pd
import numpy as np
import inspect

ORIGIN = pd.Timestamp("2025-12-31")
HORIZON = 365

# 1. Verify that the existing forecasting function is available
if "run_chronos_forecast" not in globals():
    raise NameError(
        "run_chronos_forecast() is not available. "
        "Please run the cell where this function was defined."
    )

print("Existing Chronos function:")
print(inspect.signature(run_chronos_forecast))

# 2. Generate forecast from the common origin
print("\nGenerating Chronos-2 forecast...")
print("Forecast origin:", ORIGIN.date())
print("Forecast horizon:", HORIZON, "days")

chronos_2026 = run_chronos_forecast(
    ORIGIN,
    HORIZON
)

# 3. Inspect output
if not isinstance(chronos_2026, pd.DataFrame):
    raise TypeError(
        "Expected a pandas DataFrame from run_chronos_forecast()."
    )

print("\nChronos forecast generated.")
print("Rows:", len(chronos_2026))
print("Columns:", chronos_2026.columns.tolist())

display(chronos_2026.head(10))
display(chronos_2026.tail(5))

Existing Chronos function:
(origin, horizon)

Generating Chronos-2 forecast...
Forecast origin: 2025-12-31
Forecast horizon: 365 days


TypeError: '<' not supported between instances of 'int' and 'Timestamp'


# Controlled three-model comparison — common forecast origin

The earlier rolling-origin scorecard is retained as a Chronos backtesting diagnostic. The final comparison below is separate and uses one common forecast origin (31 December 2025), the same 2026 evaluation dates, and the actual retail prices observed through 25 October 2026.

Run this section only after the controlled Chronos forecast cell has completed successfully.


In [ ]:

# ============================================================
# CONTROLLED COMPARISON — PREPARE COMMON DAILY DATA
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

ORIGIN = pd.Timestamp("2025-12-31")
EVALUATION_START = pd.Timestamp("2026-01-01")
EVALUATION_END = pd.Timestamp("2026-10-25")

# Actual AP retail prices
actual_eval = raw[["date", "AP_retail_rs_kg"]].copy()
actual_eval["date"] = pd.to_datetime(
    actual_eval["date"], errors="coerce"
).dt.normalize()
actual_eval["Actual"] = pd.to_numeric(
    actual_eval["AP_retail_rs_kg"], errors="coerce"
)
actual_eval = actual_eval.dropna(subset=["date", "Actual"])
actual_eval = actual_eval[
    actual_eval["date"].between(EVALUATION_START, EVALUATION_END)
][["date", "Actual"]]

# Chronos-2 forecast generated from the shared origin
chronos_eval = chronos_2026.copy()
chronos_eval["date"] = normalize_daily_dates(
    chronos_eval["date"]
)
chronos_eval = chronos_eval.rename(columns={
    "P50": "Chronos_Prediction",
    "P10": "Chronos_P10",
    "P90": "Chronos_P90"
})
chronos_eval = chronos_eval[
    chronos_eval["date"].between(EVALUATION_START, EVALUATION_END)
][["date", "Chronos_Prediction", "Chronos_P10", "Chronos_P90"]]

# Hybrid predictions
hybrid_eval = hybrid[["date", "Hybrid_Prediction"]].copy()
hybrid_eval["date"] = pd.to_datetime(
    hybrid_eval["date"], errors="coerce", utc=True
).dt.tz_localize(None).dt.normalize()
hybrid_eval = hybrid_eval.dropna(subset=["date"])
hybrid_eval = hybrid_eval[
    hybrid_eval["date"].between(EVALUATION_START, EVALUATION_END)
]

# Other model predictions
other_eval = excel[["date", "Other_Model_Prediction"]].copy()
other_eval["date"] = pd.to_datetime(
    other_eval["date"], errors="coerce"
).dt.normalize()
other_eval = other_eval.dropna(subset=["date"])
other_eval = other_eval[
    other_eval["date"].between(EVALUATION_START, EVALUATION_END)
]

# One row per calendar date; inner joins enforce identical evaluation dates.
controlled_comparison = (
    actual_eval
    .merge(chronos_eval, on="date", how="inner")
    .merge(hybrid_eval, on="date", how="inner")
    .merge(other_eval, on="date", how="inner")
    .sort_values("date")
    .reset_index(drop=True)
)

prediction_cols = [
    "Chronos_Prediction",
    "Hybrid_Prediction",
    "Other_Model_Prediction"
]

if controlled_comparison.empty:
    raise ValueError("No common dates across actuals and all three forecasts.")

if controlled_comparison[prediction_cols + ["Actual"]].isna().any().any():
    raise ValueError("Missing values remain in the common-date comparison.")

# Error columns
for model, pred_col in [
    ("Chronos", "Chronos_Prediction"),
    ("Hybrid", "Hybrid_Prediction"),
    ("Other_Model", "Other_Model_Prediction")
]:
    controlled_comparison[f"{model}_Error"] = (
        controlled_comparison[pred_col] - controlled_comparison["Actual"]
    )
    controlled_comparison[f"{model}_Abs_Error"] = (
        controlled_comparison[f"{model}_Error"].abs()
    )

print("CONTROLLED COMPARISON AUDIT")
print("Forecast origin:", ORIGIN.date())
print("Common actual dates:", len(controlled_comparison))
print("Evaluation period:",
      controlled_comparison["date"].min().date(),
      "to",
      controlled_comparison["date"].max().date())

display(controlled_comparison.head(10))
display(controlled_comparison.tail(10))


In [ ]:

# ============================================================
# CONTROLLED COMPARISON — FAIR METRICS ON THE SAME DATES
# ============================================================

def score_model(frame, prediction_column, model_name):
    actual_values = frame["Actual"].to_numpy(dtype=float)
    predicted_values = frame[prediction_column].to_numpy(dtype=float)
    errors = predicted_values - actual_values

    mae = np.mean(np.abs(errors))
    rmse = np.sqrt(np.mean(errors ** 2))
    nonzero = np.abs(actual_values) > 1e-12
    mape = (
        np.mean(np.abs(errors[nonzero] / actual_values[nonzero])) * 100
        if nonzero.any() else np.nan
    )

    return {
        "Model": model_name,
        "Forecast origin": ORIGIN.date(),
        "Evaluation start": frame["date"].min().date(),
        "Evaluation end": frame["date"].max().date(),
        "Observations": len(frame),
        "MAE (₹/kg)": mae,
        "RMSE (₹/kg)": rmse,
        "MAPE (%)": mape
    }

controlled_scorecard = pd.DataFrame([
    score_model(controlled_comparison, "Chronos_Prediction", "Chronos-2"),
    score_model(controlled_comparison, "Hybrid_Prediction", "Hybrid"),
    score_model(controlled_comparison, "Other_Model_Prediction", "Other Model")
]).sort_values("MAE (₹/kg)").reset_index(drop=True)

print("FINAL FAIR MODEL PERFORMANCE COMPARISON")
display(controlled_scorecard.round(3))

# Count which model was closest on each date.
abs_error_columns = {
    "Chronos-2": "Chronos_Abs_Error",
    "Hybrid": "Hybrid_Abs_Error",
    "Other Model": "Other_Model_Abs_Error"
}

error_matrix = controlled_comparison[
    list(abs_error_columns.values())
].to_numpy()

winner_labels = list(abs_error_columns.keys())
controlled_comparison["Closest_Model"] = [
    winner_labels[int(np.argmin(row))]
    for row in error_matrix
]

winner_counts = (
    controlled_comparison["Closest_Model"]
    .value_counts()
    .reindex(winner_labels, fill_value=0)
    .rename_axis("Model")
    .reset_index(name="Days with lowest absolute error")
)
winner_counts["Share of evaluated days (%)"] = (
    winner_counts["Days with lowest absolute error"]
    / len(controlled_comparison) * 100
)

print("DAILY WINNER COUNTS")
display(winner_counts.round(2))


In [ ]:

# ============================================================
# CONTROLLED COMPARISON — DAILY TABLE AND EXPORTS
# ============================================================

daily_comparison_table = controlled_comparison[[
    "date",
    "Actual",
    "Chronos_Prediction",
    "Chronos_P10",
    "Chronos_P90",
    "Hybrid_Prediction",
    "Other_Model_Prediction",
    "Chronos_Abs_Error",
    "Hybrid_Abs_Error",
    "Other_Model_Abs_Error",
    "Closest_Model"
]].copy()

daily_comparison_table = daily_comparison_table.rename(columns={
    "date": "Date",
    "Actual": "Actual Retail Price (₹/kg)",
    "Chronos_Prediction": "Chronos-2 Prediction (₹/kg)",
    "Chronos_P10": "Chronos-2 P10 (₹/kg)",
    "Chronos_P90": "Chronos-2 P90 (₹/kg)",
    "Hybrid_Prediction": "Hybrid Prediction (₹/kg)",
    "Other_Model_Prediction": "Other Model Prediction (₹/kg)",
    "Chronos_Abs_Error": "Chronos-2 Absolute Error",
    "Hybrid_Abs_Error": "Hybrid Absolute Error",
    "Other_Model_Abs_Error": "Other Model Absolute Error"
})

print("DAILY ACTUAL VS MODEL PREDICTIONS")
display(daily_comparison_table.head(30))

# Save all key comparison outputs.
daily_comparison_table.to_csv(
    "/content/Controlled_Daily_Model_Comparison.csv",
    index=False
)
controlled_scorecard.to_csv(
    "/content/Controlled_Model_Performance_Scorecard.csv",
    index=False
)
winner_counts.to_csv(
    "/content/Controlled_Daily_Winner_Counts.csv",
    index=False
)

print("Saved:")
print("/content/Controlled_Daily_Model_Comparison.csv")
print("/content/Controlled_Model_Performance_Scorecard.csv")
print("/content/Controlled_Daily_Winner_Counts.csv")
